# Approach 1: RoBERTa MLM (Masked Language Model)

This notebook documents our first approach — fine-tuning RoBERTa using 
Masked Language Modelling (MLM) for spell correction in code comments.

## Why this approach failed
MLM replaces the corrupted word with `<mask>` before feeding to the model.
This means the model never sees the actual typo — it only sees a blank.
So instead of fixing spelling, it predicts contextually plausible words:

- Input: `"<mask> the usr from database"` 
- Predicted: `"Sort"` (makes grammatical sense but ignores the typo)
- Expected: `"Returns"`

## What we switched to
We switched to **T5 (seq2seq)** — see `kaggle_t5_training.ipynb`.
T5 sees the full corrupted sentence and rewrites it correctly.

## Results from this approach
- Training accuracy: 98% (on 3,000 examples)
- Evaluation Top-1: 16.6%
- Evaluation Top-5: 38.7%

In [ ]:
#1
import torch, json, random, re, pickle
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, RobertaForMaskedLM

In [ ]:
#3
with open(r"C:\Users\thiru\autocorrect_comments\data\tokenized_data.pkl", "rb") as f:
    saved = pickle.load(f)

all_input_ids       = saved["input_ids"]
all_attention_masks = saved["attention_masks"]
all_mask_positions  = saved["mask_positions"]
all_labels          = saved["labels"]

print(f"Loaded {len(all_labels):,} examples")

In [ ]:
#4
class CommentCorrectionDataset(Dataset):
    def __init__(self, input_ids, attention_masks, mask_positions, labels):
        self.input_ids       = input_ids
        self.attention_masks = attention_masks
        self.mask_positions  = mask_positions
        self.labels          = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids"      : self.input_ids[idx],
            "attention_mask" : self.attention_masks[idx],
            "mask_position"  : self.mask_positions[idx],
            "label"          : self.labels[idx],
        }

# Use different variable names — no conflict with training loop
N = 3000
small_dataset = CommentCorrectionDataset(
    all_input_ids[:N],
    all_attention_masks[:N],
    all_mask_positions[:N],
    all_labels[:N],
)
small_dataloader = DataLoader(small_dataset, batch_size=8, shuffle=True)

print(f"Dataset size : {len(small_dataset)}")
print(f"Batches/epoch: {len(small_dataloader)}")

In [ ]:
#7
from spellchecker import SpellChecker

spell = SpellChecker()

def find_typo(comment):
    """
    Given a comment string, find the first word that looks like a typo.
    Returns: (typo_word, position) or (None, None) if no typo found
    """
    tokens = comment.split()
    
    for idx, word in enumerate(tokens):
        # Clean the word — remove punctuation before checking
        clean = word.strip(".,!?:;\"'()")
        
        # Skip short words — "a", "in", "of" are not worth checking
        if len(clean) < 4:
            continue
        
        # Skip words that look like code — camelCase, snake_case, ALL_CAPS
        if "_" in clean or clean.isupper():
            continue
        
        # Skip words starting with capital — likely proper nouns
        # except the first word of the comment
        if idx > 0 and clean[0].isupper():
            continue
        
        # Ask spell checker — is this word misspelled?
        misspelled = spell.unknown([clean.lower()])
        if misspelled:
            return clean, idx
    
    return None, None


# Test it on some examples
test_comments = [
    "Returns the usr from database by ID",
    "Computes the waighted average of all values",
    "Get all child foleder of a folder",
    "Initialize the database connection",   # no typo
]

print("TYPO DETECTION TEST")
print("=" * 50)
for comment in test_comments:
    typo, pos = find_typo(comment)
    if typo:
        print(f"Comment : {comment}")
        print(f"Typo    : '{typo}' at position {pos}")
    else:
        print(f"Comment : {comment}")
        print(f"Typo    : None found ✓")
    print()

In [ ]:
# We trained on examples 0-2999
# We evaluate on examples 3000 onwards — model never saw these

eval_dataset = CommentCorrectionDataset(
    all_input_ids[3000:4000],       # 1000 unseen examples
    all_attention_masks[3000:4000],
    all_mask_positions[3000:4000],
    all_labels[3000:4000],
)
eval_dataloader = DataLoader(eval_dataset, batch_size=8, shuffle=False)

print(f"Evaluation set: {len(eval_dataset)} examples")
print("Running evaluation...")

model.eval()    # no dropout during evaluation

top1_correct = 0
top5_correct = 0
total        = 0

with torch.no_grad():    # no gradient calculation needed
    for batch in eval_dataloader:
        
        inp_ids = batch["input_ids"].to(device)
        attn    = batch["attention_mask"].to(device)
        mpos    = batch["mask_position"]
        labs    = batch["label"].to(device)
        
        outputs = model(input_ids=inp_ids, attention_mask=attn)
        
        batch_size  = inp_ids.shape[0]
        mask_logits = torch.stack([
            outputs.logits[i, mpos[i], :]
            for i in range(batch_size)
        ])
        
        # Top-1: highest probability prediction
        top1_preds = mask_logits.argmax(dim=-1)
        top1_correct += (top1_preds == labs).sum().item()
        
        # Top-5: check if correct answer is in top 5 predictions
        top5_preds = mask_logits.topk(5, dim=-1).indices
        for i in range(batch_size):
            if labs[i].item() in top5_preds[i].tolist():
                top5_correct += 1
        
        total += batch_size

top1_accuracy = top1_correct / total * 100
top5_accuracy = top5_correct / total * 100

print(f"\n{'='*45}")
print(f"EVALUATION RESULTS (on {total} unseen examples)")
print(f"{'='*45}")
print(f"Top-1 Accuracy : {top1_accuracy:.1f}%")
print(f"Top-5 Accuracy : {top5_accuracy:.1f}%")
print(f"{'='*45}")